George Rodriguez

Dataset: https://www.kaggle.com/datasets/abdelazizsami/breast-cancer-wisconsin-diagnostic

In [ ]:
import numpy as nm
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("wdbc.data", header=None)

In [ ]:
# Add column names:
# col 0 = ID, col 1 = Diagnosis (M/B), cols 2..31 = features
df.columns = ["ID", "Diagnosis"] + [f"feature_{i}" for i in range(1, 31)]
df

In [ ]:
df = df.drop(columns=["ID"])   # ID isn't predictive
df = df.dropna()               # safety

In [ ]:
le = LabelEncoder()
df["Diagnosis"] = le.fit_transform(df["Diagnosis"])
# With LabelEncoder: usually B -> 0, M -> 1 (check le.classes_ if you want)
print("Diagnosis classes:", le.classes_)  # should print ['B', 'M']

In [ ]:
x = df.drop(columns=["Diagnosis"])
y = df["Diagnosis"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

In [ ]:
print(f"Accuracy: {acc:.4f}")
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=[f"{le.classes_[0]} (0)", f"{le.classes_[1]} (1)"]
))

In [ ]:
def get_prediction():
    cols = list(x.columns)
    while True:
        try:
            print("\nEnter the 30 feature values:")
            vals = []
            for c in cols:
                vals.append(float(input(f"Enter {c}: ")))

            input_df = pd.DataFrame([vals], columns=cols)

            # scale input like training
            input_scaled = sc.transform(input_df)

            pred_class = model.predict(input_scaled)[0]
            pred_prob_malignant = model.predict_proba(input_scaled)[0, 1]  # prob of class 1

            # Decode class back to label
            pred_label = le.inverse_transform([pred_class])[0]

            print(f"\nPredicted Diagnosis: {pred_label}")
            print(f"Probability of class '{le.inverse_transform([1])[0]}' (usually Malignant): {pred_prob_malignant:.4f}")
            break

        except ValueError:
            print("Invalid input. Please enter numeric values.")
        except Exception as e:
            print(f"An error occurred: {e}")
            break

while True:
    get_prediction()
    another = input("Do you want to predict another diagnosis? (yes/no): ")
    if another.lower() != "yes":
        break

# Summary
This program learns from a breast cancer dataset to classify whether a tumor is more likely “benign” or “malignant” based on 30 measured characteristics. It cleans the data, converts the diagnosis into a simple yes/no format, and trains a model that can recognize patterns linked to each diagnosis. It then tests itself on a reserved portion of the data and reports how often it was right and what kinds of mistakes it made (especially whether it misses malignant cases). Finally, it includes a simple input tool where someone can enter the 30 measurements and get a predicted diagnosis plus a confidence score.